# Nanopath Run Dashboard

Every xarch run's `summary.json` under `/data/$USER/nanopath`, loaded into one table. Below: final score over time (recolor by any config key that varies), then a head-to-head breakdown of two runs on the six `final_score` components (weights in `probe.SCORE_WEIGHTS`). Needs the notebook deps: `uv sync --group notebook`.

In [ ]:
import json, os
from datetime import datetime
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import Dropdown, interact

ROOT = Path(f"/data/{os.environ['USER']}/nanopath")
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]  # fixed CVD-checked categorical order
PARTS = {"Classification F1": "final_probe_classification_mean_f1", "Segmentation F1": "final_probe_seg_mean_f1",
         "Progression AUC": "final_probe_slide_mean_auc", "Mutation AUC": "final_probe_auc_mean",
         "Survival c-index": "final_probe_survival_mean_cindex", "Robustness": "final_probe_robustness_mean"}

# One row per run; nested keys flatten to "a.b" columns, lists become strings so they can be compared/grouped.
paths = sorted(ROOT.glob("*-xarch*/summary.json"))  # runs still training have no summary.json yet
df = pd.json_normalize([json.loads(p.read_text()) for p in paths]).map(lambda v: str(v) if isinstance(v, list) else v)
df["run"] = [p.parent.name for p in paths]
df["time"] = [datetime.fromtimestamp(p.stat().st_mtime) for p in paths]  # summary write time spreads same-day runs apart
df = df.rename(columns={v: k for k, v in PARTS.items()}).sort_values("time").reset_index(drop=True)

# Config settings = everything that isn't a probe result or run bookkeeping; color options are the ones that vary with <= 8 values.
SETTINGS = [c for c in df.filter(regex=r"^(?!final_|wandb\.|unique_|visible_|steps_|train_|flop|slurm|project|recipe_id|config_path|run$|time$)").columns if c not in PARTS]
COLOR_KEYS = sorted(c for c in SETTINGS if 1 < df[c].nunique(dropna=False) <= len(PALETTE))
print(f"{len(df)} runs, latest {df.run.iloc[-1]}")

In [ ]:
'Classification F1' in list(df.columns)

In [ ]:
# Final score over time, colored by a config key; missing keys (older code) show as "unset". Gray step line = best score so far.
def timeline(color_by):
    d = df.assign(**{color_by: df[color_by].fillna("unset").astype(str)})
    fig = px.scatter(d, x="time", y="final_score", color=color_by, hover_name="run", hover_data={k: ":.4f" for k in PARTS},
                     color_discrete_sequence=PALETTE, category_orders={color_by: sorted(d[color_by].unique())})
    fig.update_traces(marker=dict(size=10, line=dict(width=2, color="white")))
    fig.add_scatter(x=d.time, y=d.final_score.cummax(), mode="lines", line=dict(shape="hv", color="#888", width=2), name="best so far", hoverinfo="skip")
    fig.update_layout(template="plotly_white", height=500)
    fig.show()
interact(timeline, color_by=Dropdown(options=COLOR_KEYS, value="dino_balance.type"));

In [ ]:
# Head-to-head: per-component score deltas (B - A), the absolute values, and every config key that differs between the two runs.
best, newest = df.run[df.final_score.idxmax()], df.run.iloc[-1]
runs = list(df.run[::-1])  # newest first
def compare(a, b):
    ra, rb = df.set_index("run").loc[a], df.set_index("run").loc[b]
    keys = [*PARTS, "final_score"]
    delta = (rb[keys] - ra[keys]).astype(float)
    fig = go.Figure(go.Bar(x=keys, y=delta, marker_color=[PALETTE[0] if v >= 0 else PALETTE[1] for v in delta],
                           text=[f"{v:+.4f}" for v in delta], customdata=list(zip(ra[keys], rb[keys])),
                           hovertemplate="%{x}<br>A %{customdata[0]:.4f} → B %{customdata[1]:.4f}<extra></extra>"))
    fig.add_hrect(y0=-0.004, y1=0.004, fillcolor="#888", opacity=0.15, line_width=0)  # leader margin from AGENTS.md
    fig.update_layout(template="plotly_white", height=420, title=f"B − A:  {b}  vs  {a}", yaxis_title="Δ score")
    fig.show()
    display(pd.DataFrame({"A": ra[keys], "B": rb[keys], "Δ": delta}).astype(float).style.format("{:+.4f}", subset="Δ").format("{:.4f}", subset=["A", "B"]))
    diff = [c for c in SETTINGS if str(ra[c]) != str(rb[c])]
    display(pd.DataFrame({"A": ra[diff], "B": rb[diff]}).fillna("unset"))
interact(compare, a=Dropdown(options=runs, value=best, description="A"), b=Dropdown(options=runs, value=newest, description="B"));